# 🧪 Protein Structure Analysis

This notebook explores how to analyze 3D protein structures with **Biopython**.

**You will learn:**
1. Download structures from the PDB
2. Navigate the structure object hierarchy (Structure → Model → Chain → Residue → Atom)
3. Compute distances between atoms
4. Identify ligands and active sites
5. Compute basic structural properties


In [ ]:
from Bio.PDB import PDBParser, PDBList, NeighborSearch, Selection
from Bio.PDB.Polypeptide import is_aa
import numpy as np
import os

print("Biopython.PDB ready.")


## 1. Download a structure

We use `1AKE` (adenylate kinase) as an example — small, well-studied, and pharmacologically interesting.


In [ ]:
os.makedirs("../data/pdb", exist_ok=True)

pdbl = PDBList()
pdb_file = pdbl.retrieve_pdb_file('1AKE', pdir='../data/pdb', file_format='pdb')
print(f"File downloaded: {pdb_file}")


## 2. Parse the structure

The Biopython hierarchy is: `Structure → Model → Chain → Residue → Atom`.


In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure('1AKE', pdb_file)

print(f"Structure: {structure.id}")
print(f"Number of models: {len(structure)}")

for model in structure:
    print(f"  Model {model.id}: {len(model)} chains")
    for chain in model:
        residues = list(chain)
        amino_acids = [r for r in residues if is_aa(r)]
        print(f"    Chain {chain.id}: {len(residues)} residues, {len(amino_acids)} amino acids")


## 3. Iterate over residues and atoms

In [ ]:
# Take the first chain
model = structure[0]
chain_a = model['A']

# Show the first 5 amino acid residues
print("First 5 amino acid residues in chain A:\n")
count = 0
for residue in chain_a:
    if is_aa(residue):
        atoms = list(residue.get_atoms())
        ca = next((a for a in atoms if a.id == 'CA'), None)
        coord = ca.coord if ca is not None else 'N/A'
        print(f"  {residue.resname} {residue.id[1]:3d}  | {len(atoms)} atoms | CA: {coord}")
        count += 1
        if count >= 5:
            break


## 4. Distances between atoms

The CA-CA distance (alpha carbons) is fundamental for analyzing protein folding.


In [ ]:
# Compute all alpha carbons
ca_atoms = [residue['CA'] for residue in chain_a
            if is_aa(residue) and 'CA' in residue]
print(f"Alpha carbons in chain A: {len(ca_atoms)}")

# Distance between residue 1 and 2 (consecutive)
d1 = ca_atoms[0] - ca_atoms[1]
print(f"\nCA-CA distance residues {ca_atoms[0].get_parent().id[1]}-{ca_atoms[1].get_parent().id[1]}: {d1:.2f} Å")
print(f"(typical ~3.8 Å between consecutive residues)")

# Distance between first and last residue
dN = ca_atoms[0] - ca_atoms[-1]
print(f"\nDistance between first and last CA: {dN:.2f} Å")


## 5. Sequence from structure

In [ ]:
# 3-letter to 1-letter amino acid code
three_to_one = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C',
    'GLU':'E','GLN':'Q','GLY':'G','HIS':'H','ILE':'I',
    'LEU':'L','LYS':'K','MET':'M','PHE':'F','PRO':'P',
    'SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
}

sequence = "".join(
    three_to_one.get(residue.resname, 'X')
    for residue in chain_a
    if is_aa(residue)
)
print(f"Length: {len(sequence)}")
print(f"Sequence:\n{sequence}")


## 6. Identify HETATM (ligands and ions)

Non-standard residues are tagged in the structure with a non-blank `id[0]` field.


In [ ]:
hetero = []
for residue in chain_a:
    hetflag, resseq, icode = residue.id
    if hetflag.strip():  # not blank → HETATM
        hetero.append((hetflag, residue.resname, resseq))

print(f"Non-standard residues in chain A: {len(hetero)}")
for h in hetero[:10]:
    print(f"  {h[1]:4} (id={h[2]}, type={h[0]})")


## 7. Find residues near a ligand

The classical use case: identify the **active site** by finding residues within X Å of a ligand.

We use `NeighborSearch` for an efficient spatial query.


In [ ]:
# 1AKE has the AP5 inhibitor (residue with non-standard hetflag)
# We find atoms close to ANY ligand atom

# Collect ligand atoms (non-water HETATM)
ligand_atoms = []
for residue in chain_a:
    hetflag, _, _ = residue.id
    if hetflag.strip() and residue.resname != 'HOH':
        ligand_atoms.extend(residue.get_atoms())

print(f"Ligand atoms found: {len(ligand_atoms)}")

if ligand_atoms:
    # All protein atoms (only standard amino acids)
    protein_atoms = [a for r in chain_a if is_aa(r) for a in r.get_atoms()]
    ns = NeighborSearch(protein_atoms)

    nearby_residues = set()
    for atom in ligand_atoms:
        for nearby in ns.search(atom.coord, 5.0):  # 5 Å radius
            nearby_residues.add(nearby.get_parent())

    print(f"\nResidues within 5 Å of the ligand: {len(nearby_residues)}")
    for r in sorted(nearby_residues, key=lambda x: x.id[1]):
        print(f"  {r.resname} {r.id[1]}")


## 📝 Exercises

1. **Easy**: download another structure (e.g. `1HVR` — HIV protease) and report basic properties.
2. **Medium**: write a function that returns the longest CA-CA distance in a structure.
3. **Medium**: identify all disulfide bridges (Cys-Cys at < 2.2 Å between SG atoms).
4. **Hard**: compute the radius of gyration of the protein and compare it to typical values.
5. **Pharma-focused**: download a CYP3A4 structure (e.g. `1TQN`) and identify residues forming the heme active site.

## 📚 Resources

- [Biopython.PDB Documentation](https://biopython.org/wiki/The_Biopython_Structural_Bioinformatics_FAQ)
- [PDB101](https://pdb101.rcsb.org/) — RCSB educational portal
- [BioPandas](https://rasbt.github.io/biopandas/) — alternative based on DataFrames
